# Device Lookup Batch — Unit / Invariant Tests

Standalone tests for the cleaning functions in `device_lookup_batch_cleaning_v6.ipynb`. They run entirely on tiny in-memory fixtures — **no S3 access and no full data read required**.

> The cleaning functions below are **copied** from the cleaning notebook so this notebook runs on its own. If you change a function there, update it here too (or move the functions into a shared `.py` module and import them in both notebooks).

In [ ]:
%%pyspark project.spark.compatibility
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
import contextlib

In [ ]:
%%pyspark project.spark.compatibility
spark = SparkSession.builder.appName("device_lookup_batch_tests").getOrCreate()

## Cleaning functions (copied from the cleaning notebook — keep in sync)

In [ ]:
%%pyspark project.spark.compatibility
def flatten_json_column_with_schema(
    df: DataFrame,
    json_col: str,
    schema: StructType,
    prefix: str = None,
    drop_original: bool = False,
) -> DataFrame:
    """Parse a JSON-string column with a given schema and promote each field to its own column."""
    parsed = F.from_json(F.col(json_col), schema)
    for field in schema.fieldNames():
        output_col = f"{prefix}_{field}" if prefix else field
        df = df.withColumn(output_col, parsed[field])
    if drop_original:
        df = df.drop(json_col)
    return df

In [ ]:
%%pyspark project.spark.compatibility
# notes JSON schema for device_lookup_batch.
# correlationId may not appear in every row; will be null where absent.
notes_schema = StructType([
    StructField("tac",             StringType(), True),
    StructField("oldIMSI",         StringType(), True),
    StructField("oldIMEI",         StringType(), True),
    StructField("fromADCSnapshot", StringType(), True),
    StructField("correlationId",   StringType(), True),
])

In [ ]:
%%pyspark project.spark.compatibility
def standardize_timestamp(df, ts_col, output_col=None):
    """Parse a string timestamp (yyyy-MM-dd HH:mm:ss or yyyy-MM-dd) into TimestampType; unparseable -> null. Wall-clock time is interpreted in the session timezone (set to America/Toronto)."""
    target_col = output_col or ts_col
    parsed = F.coalesce(
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd")),
    )
    return df.withColumn(target_col, parsed)

def standardize_date(df, date_col, output_col=None):
    """Parse a string date column (yyyy-MM-dd) into DateType; unparseable -> null.
    Pre-filters to the yyyy-MM-dd pattern before to_date so bad input becomes null
    under any Spark version (no try_to_date dependency)."""
    target_col = output_col or date_col
    s = F.col(date_col)
    safe = F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}$"), s).otherwise(None)
    return df.withColumn(target_col, F.to_date(safe, "yyyy-MM-dd"))

In [ ]:
%%pyspark project.spark.compatibility
# Project bronze -> silver schema (20 columns)

def clean_str(c):
    """Trim; convert literal "\\N", empty string, and "MISSING_DATA"
    (Rogers placeholder for unknown values) to null."""
    v = F.trim(F.col(c))
    return F.when((v == "\\N") | (v == "") | (v == "MISSING_DATA"), None).otherwise(v)

def safe_long(c):
    """Cast a string column to bigint. Pure digits only; non-numeric -> null.
    LongType required: bronze `id` is bigint and values can exceed Int32 max."""
    s = clean_str(c)
    return F.when(s.rlike("^[0-9]+$"), s).otherwise(None).cast(LongType())

def to_silver(df):
    """Project the dlb frame onto the silver schema (20 columns).
    `event_date` is derived from `event_timestamp` so the partition column is
    always internally consistent with the event time; bronze `date` is kept
    alongside for source-fidelity tracking.

    Key / id handling (keep in sync with the cleaning notebook):
      - record_id       = sha2(id) surrogate (STRING 64-hex), decoupled from bronze id.
      - source_event_id = bronze id.
      - correlation_id  = real value if present, else "gen-" + 36-char UUID from id.
    """
    _h = F.sha2(clean_str("id"), 256)                                        # 64 hex
    _uuid_from_id = F.concat_ws("-",
        F.substring(_h, 1, 8),  F.substring(_h, 9, 4),  F.substring(_h, 13, 4),
        F.substring(_h, 17, 4), F.substring(_h, 21, 12),
    )                                                                        # 36-char UUID shape

    return df.select(
        F.sha2(clean_str("id"), 256).alias("record_id"),                     # #3 surrogate (was safe_long id)
        clean_str("change_key").alias("phone_number_AC_hash"),
        clean_str("change_key_ats").alias("phone_number_AT_hash"),
        F.upper(clean_str("mno")).alias("mno"),
        clean_str("imei").alias("imei"),
        clean_str("imsi").alias("imsi"),
        F.col("date"),                                                       # bronze date kept
        F.col("timestamp").alias("event_timestamp"),
        F.coalesce(                                                          # #5 real value, else gen- synthetic
            F.lower(clean_str("correlationId")),
            F.concat(F.lit("gen-"), _uuid_from_id),
        ).alias("correlation_id"),
        clean_str("related_id").alias("related_id"),                         # #7 unchanged, pending Simon
        F.upper(clean_str("status")).alias("event_type"),
        clean_str("tac").alias("tac"),
        clean_str("oldIMSI").alias("oldIMSI"),
        clean_str("oldIMEI").alias("oldIMEI"),
        F.lower(clean_str("fromADCSnapshot")).alias("fromADCSnapshot"),      # #6 standardize (lower); null preserved
        F.to_timestamp(F.lit("2026-03-16 00:00:00")).alias("ingestion_ts"),
        F.lit("MYSQL").alias("source_name"),
        safe_long("id").alias("source_event_id"),                            # #2 bronze id (was null)
        F.lit(1).cast(IntegerType()).alias("schema_version"),
        F.to_date(F.col("timestamp")).alias("event_date"),                   # derived, partition col
    )


## Test fixture

In [ ]:
%%pyspark project.spark.compatibility
# Fixture Silver frame, built from a tiny synthetic bronze sample through the SAME chain
# as production (standardize_timestamp -> standardize_date -> to_silver -> dedupe).
# Used by test_extra_fields_kept_and_originals_dropped and test_silver_schema_exact, which
# only inspect columns/schema (no row values), so a 1-row fixture is enough and needs no S3.
# The notes-parsed fields (tac/oldIMSI/...) are supplied as direct columns here, mirroring
# the post-flatten shape that to_silver consumes.
_bronze_fixture = spark.createDataFrame([
    Row(id="1", change_key="k1", change_key_ats="\\N", mno="BELL",
        imei="\\N", imsi="\\N", date="2023-07-06",
        timestamp="2023-07-06 03:18:41", related_id="\\N", status="IMEI_START",
        tac="\\N", oldIMSI="\\N", oldIMEI="\\N", fromADCSnapshot="\\N", correlationId="\\N"),
])
_p = standardize_timestamp(_bronze_fixture, "timestamp")
_p = standardize_date(_p, "date")
dlb_final = to_silver(_p).dropDuplicates()

## Unit / invariant tests

In [ ]:
%%pyspark project.spark.compatibility
# ============================================================
# Unit / Invariant tests - exercise the cleaning functions with fixtures
# Run inside the notebook, or copy this block into test_device_lookup_batch.py
# under pytest (would need a module-scope `spark` fixture).
# ============================================================

def _df(rows):
    """Build a tiny DataFrame from a list of Row objects (test fixture helper)."""
    return spark.createDataFrame(rows)

@contextlib.contextmanager
def _toronto_session():
    """Pin session.timeZone to America/Toronto for deterministic timestamp tests."""
    prev = spark.conf.get("spark.sql.session.timeZone")
    spark.conf.set("spark.sql.session.timeZone", "America/Toronto")
    try:
        yield
    finally:
        spark.conf.set("spark.sql.session.timeZone", prev)

def _ts_str(df, col_name):
    """Format timestamp via Spark (session tz) and collect, avoiding PySpark collect()-time JVM tz."""
    return [r[0] for r in df.select(
        F.date_format(F.col(col_name), "yyyy-MM-dd HH:mm:ss")
    ).collect()]

def _bronze_row(**overrides):
    """Bronze fixture template; all bronze columns present so to_silver doesn't UNRESOLVED_COLUMN."""
    base = dict(
        id="1", change_key="\\N", change_key_ats="\\N", mno="BELL",
        imei="\\N", imsi="\\N", date="2023-07-06",
        timestamp="2023-07-06 03:18:41", related_id="\\N", status="IMEI_START",
        tac="\\N", oldIMSI="\\N", oldIMEI="\\N", fromADCSnapshot="\\N", correlationId="\\N",
    )
    base.update(overrides)
    return Row(**base)

# ---------- JSON expand ----------
def test_notes_schema_keys_parsed():
    rows = [Row(notes='{"tac":"35169051","oldIMSI":"a","oldIMEI":"b","fromADCSnapshot":"true","correlationId":"c"}')]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()[0]
    assert o["tac"] == "35169051"
    assert o["oldIMSI"] == "a"
    assert o["oldIMEI"] == "b"
    assert o["fromADCSnapshot"] == "true"
    assert o["correlationId"] == "c"

def test_notes_null_empty_malformed():
    # notes = null / empty / malformed JSON / \N -> all parsed fields null
    rows = [Row(notes=None), Row(notes=""), Row(notes='{"tac":'), Row(notes="\\N")]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()
    for r in o:
        assert r["tac"] is None and r["oldIMSI"] is None and r["correlationId"] is None

def test_notes_extra_keys_ignored():
    rows = [Row(notes='{"tac":"123","foo":"bar"}')]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()[0]
    assert o["tac"] == "123" and o["oldIMSI"] is None

def test_notes_partial_keys():
    rows = [Row(notes='{"tac":"99999999"}'), Row(notes='{"correlationId":"only-corr"}')]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()
    assert o[0]["tac"] == "99999999" and o[0]["correlationId"] is None
    assert o[1]["tac"] is None and o[1]["correlationId"] == "only-corr"

# ---------- timestamp / date (no timezone shift; bronze is already EST/EDT) ----------
def test_timestamp_preserves_wall_clock_winter():
    with _toronto_session():
        out = _ts_str(
            standardize_timestamp(_df([Row(timestamp="2023-11-26 01:02:07")]), "timestamp"),
            "timestamp",
        )
        assert out == ["2023-11-26 01:02:07"]

def test_timestamp_preserves_wall_clock_summer():
    with _toronto_session():
        out = _ts_str(
            standardize_timestamp(_df([Row(timestamp="2023-07-15 12:00:00")]), "timestamp"),
            "timestamp",
        )
        assert out == ["2023-07-15 12:00:00"]

def test_timestamp_invalid_to_null():
    with _toronto_session():
        out = standardize_timestamp(
            _df([Row(timestamp="garbage"), Row(timestamp="\\N"), Row(timestamp=None)]),
            "timestamp",
        ).collect()
        for r in out:
            assert r["timestamp"] is None

def test_standardize_date():
    out = standardize_date(
        _df([Row(date="2023-07-06"), Row(date="garbage"), Row(date="\\N"), Row(date=None)]),
        "date",
    ).collect()
    assert out[0]["date"].isoformat() == "2023-07-06"
    assert out[1]["date"] is None
    assert out[2]["date"] is None
    assert out[3]["date"] is None

# ---------- dedup ----------
def test_dedupe_collapses_duplicates():
    rows = [Row(id="1", change_key="k", status="IMEI_START"),
            Row(id="1", change_key="k", status="IMEI_START")]
    assert _df(rows).dropDuplicates().count() == 1

def test_dedupe_keeps_distinct_rows():
    rows = [Row(id="1", change_key="k", status="IMEI_START"),
            Row(id="1", change_key="k", status="IMEI_END")]
    assert _df(rows).dropDuplicates().count() == 2

# ---------- silver schema / types ----------
def test_extra_fields_kept_and_originals_dropped():
    # notes-parsed fields kept; raw bronze names (id / change_key / status / notes) gone
    extras = {"tac", "oldIMSI", "oldIMEI", "fromADCSnapshot", "correlation_id"}
    assert extras <= set(dlb_final.columns)
    for raw in ["notes", "id", "change_key", "status", "change_key_ats", "correlationId"]:
        assert raw not in dlb_final.columns, raw

def test_silver_schema_exact():
    expected = [
        ("record_id", "string"),
        ("phone_number_AC_hash", "string"),
        ("phone_number_AT_hash", "string"),
        ("mno", "string"),
        ("imei", "string"),
        ("imsi", "string"),
        ("date", "date"),
        ("event_timestamp", "timestamp"),
        ("correlation_id", "string"),
        ("related_id", "string"),
        ("event_type", "string"),
        ("tac", "string"),
        ("oldIMSI", "string"),
        ("oldIMEI", "string"),
        ("fromADCSnapshot", "string"),
        ("ingestion_ts", "timestamp"),
        ("source_name", "string"),
        ("source_event_id", "bigint"),
        ("schema_version", "int"),
        ("event_date", "date"),
    ]
    got = [(f.name, f.dataType.simpleString()) for f in dlb_final.schema.fields]
    assert got == expected

def test_record_id_sha256():
    # record_id is now a sha256 surrogate (STRING, 64-hex). Clean id -> 64-hex;
    # \N / "" -> null (clean_str nulls them, sha2(null)=null). Deterministic + trims.
    import re
    rows = [
        _bronze_row(id="5128020852"),     # valid -> 64-hex
        _bronze_row(id=" 42 "),           # trimmed before hashing
        _bronze_row(id="\\N"),            # literal \N -> null
        _bronze_row(id=""),               # empty -> null
    ]
    got = to_silver(_df(rows)).collect()
    assert re.fullmatch(r"[0-9a-f]{64}", got[0]["record_id"])
    assert got[2]["record_id"] is None
    assert got[3]["record_id"] is None
    # trim + determinism: " 42 " and "42" hash to the same value
    assert got[1]["record_id"] == to_silver(_df([_bronze_row(id="42")])).collect()[0]["record_id"]

def test_event_type_uppercased():
    r = to_silver(_df([_bronze_row(status=" imei_start ")])).collect()[0]
    assert r["event_type"] == "IMEI_START"

def test_mno_uppercased():
    r = to_silver(_df([_bronze_row(mno=" telus ")])).collect()[0]
    assert r["mno"] == "TELUS"

def test_correlation_id_renamed_from_notes_field():
    # real value is preserved (coalesce keeps lower(clean_str(...)) before the gen- branch)
    r = to_silver(_df([_bronze_row(correlationId="corr-1")])).collect()[0]
    assert r["correlation_id"] == "corr-1"

def test_correlation_id_fill_when_null():
    # #5: null correlationId -> deterministic synthetic "gen-" + 36-char UUID from id.
    import re
    r = to_silver(_df([_bronze_row(correlationId="\\N")])).collect()[0]
    assert r["correlation_id"].startswith("gen-")
    # stripping "gen-" leaves a 36-char UUID shape (same format as a real value)
    assert re.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}",
                        r["correlation_id"][4:])
    # deterministic: same id -> same synthetic value across runs
    assert r["correlation_id"] == to_silver(_df([_bronze_row(correlationId="\\N")])).collect()[0]["correlation_id"]

def test_fromadcsnapshot_lowercased_null_preserved():
    # #6: standardize = lower() only; nulls are NOT filled (null stays null).
    r_true = to_silver(_df([_bronze_row(fromADCSnapshot="TRUE")])).collect()[0]
    r_null = to_silver(_df([_bronze_row(fromADCSnapshot="\\N")])).collect()[0]
    assert r_true["fromADCSnapshot"] == "true"        # uppercase canonicalized
    assert r_null["fromADCSnapshot"] is None           # \N -> null (not filled)

def test_missing_data_placeholder_to_null():
    # "MISSING_DATA" is treated identically to "\N" by clean_str across all string columns.
    r = to_silver(_df([_bronze_row(
        imei="MISSING_DATA", imsi="MISSING_DATA",
        oldIMEI="MISSING_DATA", oldIMSI="MISSING_DATA",
        tac="MISSING_DATA",
    )])).collect()[0]
    assert r["imei"]    is None
    assert r["imsi"]    is None
    assert r["oldIMEI"] is None
    assert r["oldIMSI"] is None
    assert r["tac"]     is None

def test_constants_pipeline_columns():
    r = to_silver(_df([_bronze_row(id="5128020852")])).collect()[0]
    assert r["source_name"] == "MYSQL"
    assert r["schema_version"] == 1
    assert r["source_event_id"] == 5128020852       # #2 now = bronze id (bigint); was null

def test_date_passthrough_when_already_parsed():
    # bronze `date` is preserved as-is in silver
    import datetime as dt
    df_dated = _df([_bronze_row(id="1")]).withColumn("date", F.to_date(F.lit("2023-07-06")))
    r = to_silver(df_dated).collect()[0]
    assert r["date"] == dt.date(2023, 7, 6)

def test_event_date_derived_from_timestamp():
    # event_date = to_date(event_timestamp); time portion is dropped
    import datetime as dt
    raw = _df([_bronze_row(id="1", timestamp="2023-07-06 03:18:41")])
    parsed = standardize_date(standardize_timestamp(raw, "timestamp"), "date")
    r = to_silver(parsed).collect()[0]
    assert r["event_date"]      == dt.date(2023, 7, 6)
    assert r["event_timestamp"] is not None       # still carries the full timestamp


In [ ]:
%%pyspark project.spark.compatibility
# Run all unit tests
_tests = [
    test_notes_schema_keys_parsed,
    test_notes_null_empty_malformed,
    test_notes_extra_keys_ignored,
    test_notes_partial_keys,
    test_timestamp_preserves_wall_clock_winter,
    test_timestamp_preserves_wall_clock_summer,
    test_timestamp_invalid_to_null,
    test_standardize_date,
    test_dedupe_collapses_duplicates,
    test_dedupe_keeps_distinct_rows,
    test_extra_fields_kept_and_originals_dropped,
    test_silver_schema_exact,
    test_record_id_sha256,
    test_event_type_uppercased,
    test_mno_uppercased,
    test_correlation_id_renamed_from_notes_field,
    test_correlation_id_fill_when_null,
    test_fromadcsnapshot_lowercased_null_preserved,
    test_missing_data_placeholder_to_null,
    test_constants_pipeline_columns,
    test_date_passthrough_when_already_parsed,
    test_event_date_derived_from_timestamp,
]
_failed = 0
for _t in _tests:
    try:
        _t(); print("PASS  " + _t.__name__)
    except AssertionError as e:
        _failed += 1; print("FAIL  " + _t.__name__ + ": " + str(e))
    except Exception as e:
        _failed += 1; print("ERROR " + _t.__name__ + ": " + type(e).__name__ + ": " + str(e))
print()
print(str(len(_tests) - _failed) + "/" + str(len(_tests)) + " passed")
